# 🎲 Bloque 3: Modelos Probabilísticos y Bayesianos

**Objetivo:** Entender inferencia bayesiana, Hidden Markov Models y Pyro — el bloque más diferenciador del rol.

---

## 1. ¿Por qué modelos probabilísticos?

Los modelos clásicos de ML dan una predicción puntual: *"esta imagen es un gato"*.

Los modelos probabilísticos dan una **distribución** sobre la predicción: *"hay un 85% de probabilidad de que sea un gato, con incertidumbre ±10%"*.

Esto es crucial para:
- **Cuantificar incertidumbre** (ej: diagnóstico médico)
- **Datasets pequeños** donde los modelos DL no tienen suficientes datos
- **Incorporar conocimiento previo** (prior) sobre el problema

---

## 2. Probabilidad básica — repaso rápido

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# Distribuciones básicas
x = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Normal
axes[0].plot(x, stats.norm.pdf(x, loc=0, scale=1), color='steelblue')
axes[0].set_title('Normal(μ=0, σ=1)')
axes[0].fill_between(x, stats.norm.pdf(x), alpha=0.3, color='steelblue')

# Beta
x_beta = np.linspace(0, 1, 300)
axes[1].plot(x_beta, stats.beta.pdf(x_beta, 2, 5), color='coral')
axes[1].set_title('Beta(α=2, β=5)')
axes[1].fill_between(x_beta, stats.beta.pdf(x_beta, 2, 5), alpha=0.3, color='coral')

# Poisson
k = np.arange(0, 15)
axes[2].bar(k, stats.poisson.pmf(k, mu=3), color='green', alpha=0.7)
axes[2].set_title('Poisson(λ=3)')

plt.tight_layout()
plt.show()

## 3. Teorema de Bayes — el corazón de todo

$$P(\theta | D) = \frac{P(D | \theta) \cdot P(\theta)}{P(D)}$$

| Término | Nombre | Significado |
|---|---|---|
| $P(\theta)$ | **Prior** | Lo que creemos antes de ver datos |
| $P(D|\theta)$ | **Likelihood** | Probabilidad de los datos dado un parámetro |
| $P(\theta|D)$ | **Posterior** | Lo que creemos después de ver los datos |
| $P(D)$ | **Evidencia** | Constante normalizadora |

In [ ]:
# Ejemplo: estimar la probabilidad de que una moneda sea justa
# Lanzamos la moneda 10 veces y obtenemos 7 caras

n_lanzamientos = 10
n_caras = 7

theta = np.linspace(0, 1, 500)  # posibles valores de P(cara)

# Prior: creemos que la moneda es justa -> Beta(2,2) centrada en 0.5
prior = stats.beta.pdf(theta, 2, 2)

# Likelihood: probabilidad de ver 7 caras en 10 lanzamientos
likelihood = stats.binom.pmf(n_caras, n_lanzamientos, theta)

# Posterior (analítico para distribución conjugada Beta-Binomial)
# Si prior = Beta(a, b) y observamos k éxitos en n intentos:
# posterior = Beta(a + k, b + n - k)
posterior = stats.beta.pdf(theta, 2 + n_caras, 2 + n_lanzamientos - n_caras)

plt.figure(figsize=(9, 4))
plt.plot(theta, prior / prior.max(), label='Prior Beta(2,2)', linestyle='--', color='gray')
plt.plot(theta, likelihood / likelihood.max(), label='Likelihood', linestyle=':', color='coral')
plt.plot(theta, posterior / posterior.max(), label='Posterior Beta(9,5)', color='steelblue', linewidth=2)
plt.axvline(0.5, color='black', linestyle='-', alpha=0.3, label='Moneda justa (0.5)')
plt.xlabel('θ = P(cara)')
plt.title('Inferencia Bayesiana: actualizar creencias con datos')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Media posterior: {(2 + n_caras) / (2 + n_caras + 2 + n_lanzamientos - n_caras):.3f}")
print("El posterior se mueve hacia 0.7 pero el prior 'tira' hacia 0.5")

## 4. Hidden Markov Models (HMM)

Un HMM modela **secuencias** donde hay estados ocultos que generan observaciones.

**Ejemplo clásico:** el tiempo meteorológico.
- **Estados ocultos**: {Soleado, Lluvioso} — no los observamos directamente
- **Observaciones**: {Paraguas, No paraguas} — lo que vemos

Tres matrices definen un HMM:
1. **Transition matrix**: P(estado_t | estado_{t-1})
2. **Emission matrix**: P(observación | estado)
3. **Initial probabilities**: P(estado_0)

In [ ]:
# !pip install hmmlearn

In [ ]:
import numpy as np

# HMM manual: tiempo meteorológico
# Estados: 0=Soleado, 1=Lluvioso
# Observaciones: 0=No paraguas, 1=Paraguas

# Matriz de transición: P(estado_t | estado_{t-1})
# De Soleado: 70% sigue Soleado, 30% cambia a Lluvioso
# De Lluvioso: 40% cambia a Soleado, 60% sigue Lluvioso
trans = np.array([[0.7, 0.3],
                  [0.4, 0.6]])

# Matriz de emisión: P(observación | estado)
# Si Soleado: 90% No paraguas, 10% Paraguas
# Si Lluvioso: 20% No paraguas, 80% Paraguas
emit = np.array([[0.9, 0.1],
                 [0.2, 0.8]])

# Probabilidades iniciales
init = np.array([0.6, 0.4])  # 60% empieza Soleado

# Simular una secuencia de 10 días
np.random.seed(42)
n_days = 10
estados   = []
observaciones = []

estado_actual = np.random.choice(2, p=init)
for _ in range(n_days):
    estados.append(estado_actual)
    obs = np.random.choice(2, p=emit[estado_actual])
    observaciones.append(obs)
    estado_actual = np.random.choice(2, p=trans[estado_actual])

estado_nombres = ['Soleado', 'Lluvioso']
obs_nombres    = ['No paraguas', 'Paraguas']

print("Día | Estado oculto | Observación")
print("-" * 40)
for i, (e, o) in enumerate(zip(estados, observaciones)):
    print(f" {i+1:2d} | {estado_nombres[e]:<13} | {obs_nombres[o]}")

## 5. Pyro — programación probabilística sobre PyTorch

Pyro es el framework clave del rol. Permite definir modelos probabilísticos complejos y hacer inferencia variacional automáticamente.

In [ ]:
# !pip install pyro-ppl

In [ ]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

pyro.clear_param_store()
print(f"Pyro version: {pyro.__version__}")

# --------------------------------------------------------
# Ejemplo: inferir la media de una distribución Normal
# Tenemos datos y queremos saber de qué distribución vienen
# --------------------------------------------------------

# Datos observados (generados con media=5)
torch.manual_seed(42)
datos = torch.tensor([4.8, 5.2, 5.1, 4.9, 5.3, 4.7, 5.0, 5.4, 4.6, 5.1])
print(f"Datos: {datos.numpy()}")
print(f"Media muestral: {datos.mean():.3f}")

# ---- Modelo generativo ----
# Especificamos cómo creemos que se generaron los datos
def model(datos):
    # Prior sobre la media: creemos que está cerca de 0 con std=10
    mu = pyro.sample('mu', dist.Normal(0., 10.))
    
    # Likelihood: los datos se generaron con esa media y std=1
    with pyro.plate('observations', len(datos)):
        pyro.sample('obs', dist.Normal(mu, 1.), obs=datos)

# ---- Guía variacional (aproximación al posterior) ----
def guide(datos):
    # Parámetros que vamos a aprender
    mu_loc   = pyro.param('mu_loc',   torch.tensor(0.))
    mu_scale = pyro.param('mu_scale', torch.tensor(1.), constraint=dist.constraints.positive)
    
    pyro.sample('mu', dist.Normal(mu_loc, mu_scale))

In [ ]:
# Inferencia con SVI (Stochastic Variational Inference)
pyro.clear_param_store()

optimizer = Adam({'lr': 0.01})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

losses = []
for step in range(500):
    loss = svi.step(datos)
    losses.append(loss)
    if (step + 1) % 100 == 0:
        print(f"Step {step+1:4d} | ELBO loss: {loss:.4f}")

# Resultado
mu_posterior = pyro.param('mu_loc').item()
mu_std       = pyro.param('mu_scale').item()
print(f"\nPosterior → μ = {mu_posterior:.3f} ± {mu_std:.3f}")
print(f"Media real de los datos: {datos.mean():.3f}")

In [ ]:
# Visualizar convergencia
plt.figure(figsize=(8, 3))
plt.plot(losses, color='coral')
plt.title('Convergencia de SVI (ELBO loss)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Conceptos clave para recordar

| Concepto | Descripción |
|---|---|
| **Prior** | Creencias antes de ver los datos |
| **Likelihood** | Probabilidad de los datos dado el modelo |
| **Posterior** | Creencias actualizadas después de los datos |
| **HMM** | Modelo de secuencias con estados ocultos |
| **SVI** | Inferencia variacional estocástica (aproxima el posterior) |
| **ELBO** | Evidence Lower BOund — la función que minimiza SVI |
| **Guide** | En Pyro: la aproximación al posterior que queremos aprender |

---

## ✅ Resumen del bloque

- Entiendes el **Teorema de Bayes** y cómo actualizar creencias
- Conoces las **distribuciones** más comunes y cuándo usarlas
- Implementaste un **HMM** para modelar secuencias con estados ocultos
- Usaste **Pyro** para definir un modelo generativo e inferir parámetros
- Entiendes qué es **SVI y ELBO**

---

## ➡️ Siguiente paso

Continúa con el **Bloque 4: Transformers y HuggingFace** → `04_transformers_huggingface.ipynb`